# Anthropic Claude via APIM AI Gateway

This notebook tests Claude models hosted on Azure AI Foundry and proxied through Azure API Management.

## Setup

1. Deploy `options-infra/ai-gateway-anthropic` with `azd up`
2. In this directory, run `uv sync`
3. In VSCode: `Ctrl+Shift+P` → **Python: Select Interpreter** → pick `./agents_anthropic/.venv/bin/python`
4. Copy `.env.example` to `.env` and fill in your values

In [ ]:
import os
import json
import requests
import anthropic
from dotenv import load_dotenv

load_dotenv(override=True)

APIM_GATEWAY_URL    = os.environ["APIM_GATEWAY_URL"].rstrip("/")
APIM_SUBSCRIPTION_KEY = os.environ["APIM_SUBSCRIPTION_KEY"]
ANTHROPIC_MODEL     = os.environ.get("ANTHROPIC_MODEL", "claude-opus-4-6")

ANTHROPIC_BASE_URL  = f"{APIM_GATEWAY_URL}/inference/anthropic"
MESSAGES_URL        = f"{ANTHROPIC_BASE_URL}/v1/messages"
APIM_HEADERS        = {"api-key": APIM_SUBSCRIPTION_KEY}

print(f"Gateway : {APIM_GATEWAY_URL}")
print(f"Endpoint: {MESSAGES_URL}")
print(f"Model   : {ANTHROPIC_MODEL}")
print(f"Api Key  : {'*' * len(APIM_SUBSCRIPTION_KEY)}")

## 1 · Raw HTTP request

Lowest-level test — confirms APIM is reachable and the API key flow works end-to-end.

In [ ]:
payload = {
    "model": ANTHROPIC_MODEL,
    "max_tokens": 256,
    "temperature": 0.7,
    "messages": [{"role": "user", "content": "Say hello and tell me what model you are."}],
    "system": "You are a helpful assistant that provides concise answers to the user's questions."
}

resp = requests.post(
    MESSAGES_URL,
    headers={**APIM_HEADERS, "anthropic-version": "2023-06-01"},
    json=payload,
    timeout=30,
)

print(f"Status : {resp.status_code}")
print(f"Headers: x-request-id={resp.headers.get('x-request-id', 'n/a')}")

body = resp.json()
print(json.dumps(body, indent=2))
print(resp.headers)

## 2 · Anthropic Python SDK

Uses the official `anthropic` package pointed at the APIM gateway.
The APIM subscription key replaces the normal Anthropic API key.

In [ ]:
client = anthropic.Anthropic(
    base_url=ANTHROPIC_BASE_URL,
    api_key=APIM_SUBSCRIPTION_KEY,   # APIM subscription key, NOT your Anthropic API key
    default_headers=APIM_HEADERS,
)

message = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=512,
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."},
    ],
)

print(f"Stop reason  : {message.stop_reason}")
print(f"Input tokens : {message.usage.input_tokens}")
print(f"Output tokens: {message.usage.output_tokens}")
print()
print(message.content[0].text)

## 3 · System prompt + multi-turn conversation

In [ ]:
conversation = [
    {"role": "user", "content": "My name is Alex. Remember it."},
]

turn1 = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=256,
    system="You are a helpful assistant with a great memory.",
    messages=conversation,
)
print("Turn 1:", turn1.content[0].text)

# Add assistant reply and follow-up
conversation.append({"role": "assistant", "content": turn1.content[0].text})
conversation.append({"role": "user", "content": "What's my name?"})

turn2 = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=256,
    system="You are a helpful assistant with a great memory.",
    messages=conversation,
)
print("Turn 2:", turn2.content[0].text)

## 4 · Streaming response

In [ ]:
print("Streaming response:\n")

with client.messages.stream(
    model=ANTHROPIC_MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": "Count from 1 to 10, one number per line."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n\n[stream complete]")

### 4b · Raw HTTP SSE streaming

Parsing the raw server-sent-event stream from APIM — shows the exact Anthropic SSE wire format.

In [ ]:
import sseclient  # pip: sseclient-py

sse_resp = requests.post(
    MESSAGES_URL,
    headers={**APIM_HEADERS, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
    json={
        "model": ANTHROPIC_MODEL,
        "max_tokens": 128,
        "stream": True,
        "messages": [{"role": "user", "content": "Count to 5, one number per line."}],
    },
    stream=True,
)

print(f"Status: {sse_resp.status_code}")
print(f"Content-Type: {sse_resp.headers.get('content-type')}\n")

full_text = []
for event in sseclient.SSEClient(sse_resp).events():
    if event.data == "[DONE]" or not event.data:
        continue
    data = json.loads(event.data)
    etype = data.get("type", "")
    if etype == "content_block_delta":
        chunk = data["delta"].get("text", "")
        full_text.append(chunk)
        print(chunk, end="", flush=True)
    elif etype == "message_start":
        print(f"[message_start] model={data['message']['model']}")
    elif etype in ("message_delta", "message_stop"):
        print(f"\n[{etype}]", data.get("delta") or "")

print("\n\nFull text:", "".join(full_text))


## 5 · Vision (image input)

Claude supports image inputs via base64. Here we send a small test PNG.

In [ ]:
import base64, io

# Tiny 1×1 red PNG for smoke-testing vision input
red_pixel_b64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwADhQGAWjR9awAAAABJRU5ErkJggg=="
)

message = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=128,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": red_pixel_b64,
                    },
                },
                {"type": "text", "text": "What colour is the pixel in this image?"},
            ],
        }
    ],
)
print(message.content[0].text)

## 6 · APIM diagnostics

Check response headers to confirm the request was handled by APIM and inspect rate-limit state.

In [ ]:
resp = requests.post(
    MESSAGES_URL,
    headers={**APIM_HEADERS, "anthropic-version": "2023-06-01"},
    json={
        "model": ANTHROPIC_MODEL,
        "max_tokens": 64,
        "messages": [{"role": "user", "content": "ping"}],
    },
    timeout=30,
)

interesting_headers = [
    "x-request-id",
    "x-ms-region",
    "x-ratelimit-limit-requests",
    "x-ratelimit-remaining-requests",
    "x-ratelimit-limit-tokens",
    "x-ratelimit-remaining-tokens",
    "apim-request-id",
]

print(f"Status: {resp.status_code}\n")
print("Response headers:")
for h in interesting_headers:
    val = resp.headers.get(h)
    if val:
        print(f"  {h}: {val}")

print("\nAll response headers:")
for k, v in resp.headers.items():
    print(f"  {k}: {v}")

## 7 · Foundry Agent with Claude

Requires `AZURE_AI_FOUNDRY_CONNECTION_STRING` in `.env`.  
The Anthropic APIM connection is registered automatically by `azd up` — the agent uses it to route Claude calls through APIM.

Cells in this section:
- **7a** — Connect to project & inspect connections
- **7b** — Create a Claude agent
- **7c** — Single-turn question
- **7d** — Multi-turn conversation
- **7e** — Agent with tools (code interpreter)
- **7f** — Cleanup

### 7a · Connect and find the Anthropic APIM connection

In [ ]:
from azure.ai.projects.aio import AIProjectClient as AsyncAIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity.aio import DefaultAzureCredential as AsyncDefaultAzureCredential

connection_string = os.environ.get("AZURE_AI_FOUNDRY_CONNECTION_STRING", "")
if not connection_string:
    raise EnvironmentError("AZURE_AI_FOUNDRY_CONNECTION_STRING not set in .env")

aio_creds = AsyncDefaultAzureCredential()
project = AsyncAIProjectClient(endpoint=connection_string, credential=aio_creds)

anthropic_connection = None
print("Connections:")
async for conn in project.connections.list():
    print(f"  {conn.name:55s} type={conn.connection_type}")
    if conn.connection_type == "ApiManagement" and "anthropic" in conn.name.lower():
        anthropic_connection = conn.name
        print(f"  └─ ✅ Using this for agents")

if not anthropic_connection:
    raise RuntimeError("No Anthropic ApiManagement connection found in this project")

AGENT_MODEL = f"{anthropic_connection}/{ANTHROPIC_MODEL}"
print(f"\nAgent model string: {AGENT_MODEL}")


### 7b · Create a v2 agent

In [ ]:
AGENT_NAME = "claude-anthropic-gateway-agent"

# Delete existing agent with the same name to avoid version accumulation
async for existing in project.agents.list():
    if existing.name == AGENT_NAME:
        await project.agents.delete(agent_name=AGENT_NAME)
        print(f"Deleted existing agent '{AGENT_NAME}'")
        break

agent = await project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=AGENT_MODEL,
        instructions="You are a helpful assistant powered by Claude via Azure AI Foundry and APIM. Be concise.",
    ),
)
print(f"Created agent  name={agent.name}  version={agent.version}  model={agent.definition.model}")


### 7c · Single-turn question

In [ ]:
openai_client = project.get_openai_client()

conversation = await openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "What are the three primary colours? One sentence."}],
)

response = await openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="",
)
print("Agent:", response.output_text)
assert response.output_text, "Expected non-empty reply"
print("✅ Single-turn agent call succeeded")


### 7d · Multi-turn conversation on a single thread

In [ ]:
turns = [
    "My name is Alex. Remember it.",
    "What is the capital of France?",
    "What is my name?",
]

conversation = await openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": turns[0]}],
)

for i, user_msg in enumerate(turns):
    response = await openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input=user_msg if i > 0 else "",
    )
    print(f"User : {user_msg}")
    print(f"Agent: {response.output_text}")
    print()


### 7e · Streaming response

In [ ]:
conversation = await openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "Tell me hi in 5 random languages."}],
)

stream = await openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="",
    stream=True,
)

print("Streaming: ", end="")
async for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print(f"\n✅ Done  usage={event.response.to_dict()['usage']}")


### 7f · Cleanup

In [ ]:
await project.agents.delete(agent_name=AGENT_NAME)
print(f"Deleted agent '{AGENT_NAME}'")
await project.close()
await aio_creds.close()
